# Dedicated MX E4M3/A32 — resultaten van de vijfepochproef

Dit notebook analyseert **uitsluitend opgeslagen resultaten** van de dedicated E4M3-gewichtsproef: CNN-8/SVHN train-only, A32-activaties en een FP32-updatemaster. Er wordt geen training gestart en geen checkpoint geladen.

De grafieken tonen de **laatste gemeten epoch**, het **beste E4M3-checkpoint** en het vaste **epoch100-eindpunt zodra dat werkelijk beschikbaar is**. Bij deze pilot is de laatste epoch normaal 5. Ontbrekende eindaccuracies worden niet voorspeld. Synthetische controlegegevens worden bovenaan en in iedere figuurtitel herkenbaar gemarkeerd.


In [ ]:
from pathlib import Path
RUN_DIR = Path('C:\\Users\\renzo\\master-thesis-mx\\mx_runs\\mx_e4m3_dedicated_pilot5\\20261022_184112_866638_mx_run')
SAVE_FIGURES = True  # Nieuwe analysemap; ruwe resultaten blijven behouden.


In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:  # Ook analyse vanuit een gewone Python-omgeving blijft mogelijk.
    class Markdown(str):
        pass
    def display(value):
        print(value)

sys.path.insert(0, str(RUN_DIR.expanduser().resolve()))
import notebook_support as ns

run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Checkpointselectie en vergelijking

**Beste checkpoint:** het eerste strikte maximum van de opgeslagen E4M3-testaccuracy. Een gelijke score vervangt het eerdere beste checkpoint niet. Alle hieronder vermelde scores komen uit dezelfde dedicated run. De selectiebron en checkpointnaam staan expliciet in de tabel; een ontbrekend selectiebestand wordt niet stilzwijgend als bevestigde selectie behandeld.

De officiële testset wordt volgens het bestaande protocol gebruikt voor selectie. De beste score is daarom **geen onafhankelijke testschatting na selectie**. Bij een volledige latere run blijft epoch100 het vaste hoofdvergelijkingspunt; de beste epoch wordt apart getoond.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'CE-loss': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


In [ ]:
display(Markdown('## Leercurves\n\nTrainaccuracy en trainloss zijn tijdens de updates verzameld; testmetrics zijn na de epoch met bevroren evaluatiestaat gemeten. De figuur toont uitsluitend gemeten epochs.'))
figures['learning_curves'] = ns.plot_learning(run)
display(figures['learning_curves'])
plt.close(figures['learning_curves'])


In [ ]:
display(Markdown('## Looptijd en schatting voor een nieuwe training van 100 epochs'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nToegewezen en gereserveerd geheugen zijn de opgeslagen pieken van de PyTorch CUDA-allocator. Dit is geen meting van alle GPU-processen en geen bewijs van compacte native MX-uitvoering. Ontbrekende metingen blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabel rapporteert bestaande controles uit de run; het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates per epoch en doellaag'))
display(ns.weight_update_table(run))


## Betekenis van deze pilot

Vijf epochs dienen voor de technische werking, het eerste leergedrag en een looptijdmeting. Ze stellen de uiteindelijke accuracy, het gedrag na de LR-dalingen of de winst van exponenttruncatie nog niet vast. Deze dedicated proef bevat geen distillatie of truncatie; latere gezamenlijke en truncatieproeven krijgen eigen resultaten.

Een volledige training begint later opnieuw vanaf initialisatie. Dit notebook biedt daarvoor geen trainingsopdracht. Bij een onderbroken of mislukte run blijven eventueel beschikbare epochs zichtbaar, samen met de runstatus en meldingen bovenaan.

Met `SAVE_FIGURES = True` worden alleen de vier figuren en een analysemanifest naar een **nieuwe** map geschreven. Een volgende uitvoering maakt opnieuw een nieuwe map. Geen JSON, CSV, checkpoint of eerdere figuur wordt overschreven.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')

print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [path.name for path in run.input_files]}))
